In [0]:
from pyspark.sql.functions import col

catalog = "databricks_7405612194732360"
bronze_schema = "bronze_layer"
silver_schema = "aqi_silver_layer_new"

tables = [
    "bronze_city_day", 
    "bronze_city_hour", 
    "bronze_station_day", 
    "bronze_station_hour"
]

print("🚀 INITIALIZING DATA QUALITY KPI REPORT...\n")

for raw_table in tables:
    silver_table = raw_table.replace("bronze_", "")
    
    try:
        # 1. Read the existing tables (Read-Only)
        df_bronze = spark.read.table(f"{catalog}.{bronze_schema}.{raw_table}")
        df_clean = spark.read.table(f"{catalog}.{silver_schema}.{silver_table}")
        df_quarantine = spark.read.table(f"{catalog}.{silver_schema}.quarantine_{silver_table}")
        
        # 2. Gather Base Metrics
        count_initial = df_bronze.count()
        count_clean = df_clean.count()
        
        print("="*55)
        print(f"📊 KPI REPORT FOR: {silver_table.upper()}")
        print("="*55)

        # ---------------------------------------------------------
        # KPI 3: Duplicate Record Rate (Target: < 1%)
        # ---------------------------------------------------------
        count_duplicates = df_quarantine.filter(col("quarantine_reason").contains("Exact Duplicate Row")).count()
        duplicate_rate = (count_duplicates / count_initial) * 100 if count_initial > 0 else 0
        dup_status = "✅ PASS" if duplicate_rate < 1.0 else "❌ FAIL"
        
        print(f"KPI 3 | Duplicate Record Rate : {duplicate_rate:.2f}% (Target: < 1%) {dup_status}")
        print(f"      | Details: {count_duplicates} duplicates out of {count_initial} raw rows.\n")

        # ---------------------------------------------------------
        # KPI 4: Schema Consistency (Target: > 99%)
        # ---------------------------------------------------------
        count_schema_failures = df_quarantine.filter(col("quarantine_reason").contains("Invalid Data Type")).count()
        schema_consistency = ((count_initial - count_schema_failures) / count_initial) * 100 if count_initial > 0 else 100
        schema_status = "✅ PASS" if schema_consistency > 99.0 else "❌ FAIL"
        
        print(f"KPI 4 | Schema Consistency    : {schema_consistency:.2f}% (Target: > 99%) {schema_status}")
        print(f"      | Details: {count_initial - count_schema_failures} rows matched schema out of {count_initial}.\n")

        print("-" * 55 + "\n")

    except Exception as e:
        print(f"⚠️ Could not generate report for {silver_table}. Ensure tables exist. Error: {e}\n")

In [0]:
spark.table("databricks_7405612194732360.aqi_gold_layer.top_10_polluted_cities") \
    .write.mode("overwrite") \
    .parquet("/Volumes/databricks_7405612194732360/aqi_gold_layer/aqi_export/top_10_polluted_cities")

spark.table("databricks_7405612194732360.aqi_gold_layer.state_pollution_summary") \
    .write.mode("overwrite") \
    .parquet("/Volumes/databricks_7405612194732360/aqi_gold_layer/aqi_export/state_pollution_summary")

spark.table("databricks_7405612194732360.aqi_gold_layer.pollution_trend_time_series") \
    .write.mode("overwrite") \
    .parquet("/Volumes/databricks_7405612194732360/aqi_gold_layer/aqi_export/pollution_trend_time_series")

spark.table("databricks_7405612194732360.aqi_gold_layer.pollutant_trend_by_city") \
    .write.mode("overwrite") \
    .parquet("/Volumes/databricks_7405612194732360/aqi_gold_layer/aqi_export/pollutant_trend_by_city")

spark.table("databricks_7405612194732360.aqi_gold_layer.aqi_bucket_distribution") \
    .write.mode("overwrite") \
    .parquet("/Volumes/databricks_7405612194732360/aqi_gold_layer/aqi_export/aqi_bucket_distribution")

spark.table("databricks_7405612194732360.aqi_gold_layer.city_pollution_rank_over_time") \
    .write.mode("overwrite") \
    .parquet("/Volumes/databricks_7405612194732360/aqi_gold_layer/aqi_export/city_pollution_rank_over_time")